ARTI308 - Machine Learning

# Linear Regression

In this lab, we will apply **Linear Regression** to predict password strength using engineered features extracted from raw password strings.

The dataset contains the following columns:

* `password`: The raw password string
* `strength`: Password strength score (0 = Weak, 1 = Medium, 2 = Strong)

Since the model requires numerical input, we will engineer features from the raw password text.

**Let's get started!**

## Check out the data

### Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

### Check out the Data

In [ ]:
df = pd.read_csv('data.csv', on_bad_lines='skip')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.columns

## Feature Engineering

Since the only input column is a raw text string, we cannot use it directly in a regression model.
We extract meaningful numerical features from each password.

In [ ]:
df['password'] = df['password'].astype(str)

df['length']      = df['password'].str.len()
df['num_digits']  = df['password'].str.count(r'[0-9]')
df['num_upper']   = df['password'].str.count(r'[A-Z]')
df['num_lower']   = df['password'].str.count(r'[a-z]')
df['num_special'] = df['password'].str.count(r'[^a-zA-Z0-9]')
df['digit_ratio']   = df['num_digits']  / df['length']
df['upper_ratio']   = df['num_upper']   / df['length']
df['special_ratio'] = df['num_special'] / df['length']

df.head()

# EDA

Let's create some simple plots to check out the data!

In [ ]:
feature_cols = ['length', 'num_digits', 'num_upper', 'num_lower', 'num_special',
                'digit_ratio', 'upper_ratio', 'special_ratio', 'strength']

sns.pairplot(df[feature_cols].sample(2000, random_state=42))

In [ ]:
sns.histplot(df['strength'])

In [ ]:
sns.heatmap(df[feature_cols].corr(), annot=True, cmap='coolwarm')

## Training a Linear Regression Model

We will split the data into an X array containing the engineered features, and a y array with the target variable `strength`.

### X and y arrays

In [ ]:
X = df[['length', 'num_digits', 'num_upper', 'num_lower', 'num_special',
        'digit_ratio', 'upper_ratio', 'special_ratio']]
y = df['strength']

## Train Test Split

Now let's split the data into a training set and a testing set.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=101)

## Creating and Training the Model

In [ ]:
from sklearn.linear_model import LinearRegression

lm = LinearRegression()
lm.fit(X_train, y_train)

## Model Evaluation

Let's evaluate the model by checking out its coefficients and how we can interpret them.

In [ ]:
# print the intercept
print(lm.intercept_)

In [ ]:
coeff_df = pd.DataFrame(lm.coef_, X.columns, columns=['Coefficient'])
coeff_df

Interpreting the coefficients:

- Holding all other features fixed, a 1 unit increase in **length** is associated with a change in predicted strength.
- Holding all other features fixed, a 1 unit increase in **num_upper** is associated with a change in predicted strength.
- Holding all other features fixed, a 1 unit increase in **num_special** is associated with a change in predicted strength.

A positive coefficient means that feature increases the predicted strength score, while a negative coefficient decreases it.

## Predictions from our Model

Let's grab predictions off our test set and see how well it did!

In [ ]:
predictions = lm.predict(X_test)

plt.scatter(y_test, predictions)
plt.xlabel("Actual Strength")
plt.ylabel("Predicted Strength")
plt.title("Actual vs Predicted Password Strength")
plt.show()

**Residual Histogram**

In [ ]:
sns.histplot((y_test - predictions), bins=50)
plt.title("Residuals Distribution")
plt.xlabel("Residual")
plt.show()

## Regression Evaluation Metrics

Here are three common evaluation metrics for regression problems:

**Mean Absolute Error** (MAE) is the mean of the absolute value of the errors:

$$\frac 1n\sum_{i=1}^n|y_i-\hat{y}_i|$$

**Mean Squared Error** (MSE) is the mean of the squared errors:

$$\frac 1n\sum_{i=1}^n(y_i-\hat{y}_i)^2$$

**Root Mean Squared Error** (RMSE) is the square root of the mean of the squared errors:

$$\sqrt{\frac 1n\sum_{i=1}^n(y_i-\hat{y}_i)^2}$$

Comparing these metrics:

- **MAE** is the easiest to understand, because it's the average error.
- **MSE** is more popular than MAE, because MSE "punishes" larger errors, which tends to be useful in the real world.
- **RMSE** is even more popular than MSE, because RMSE is interpretable in the "y" units.

In [ ]:
from sklearn import metrics

print('MAE:', metrics.mean_absolute_error(y_test, predictions))
print('MSE:', metrics.mean_squared_error(y_test, predictions))
print('RMSE:', np.sqrt(metrics.mean_squared_error(y_test, predictions)))

### Task:
In this task, you will apply what you have learned in this lab on the same dataset but with different features.

1. Load the dataset into a DataFrame
2. Explore the data (head, info, describe)
3. Perform basic data cleaning if needed
4. Apply feature engineering (try adding new features like `has_digit`, `has_upper`, `has_special` flags)
5. Prepare the data for modeling
6. Train a Linear Regression model
7. Evaluate the model performance and compare MAE, MSE, RMSE with the results above